# Parquet exploration (scratch)

This notebook assumes the working directory is the repo root.

Files:
- `data/processed/financials_<YYYY-MM-DD>.parquet` (snapshot)
- `data/processed/financials_timeseries_<YYYY-MM-DD>.xlsx` (timeseries)


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="darkgrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)

In [ ]:
processed_dir = Path("data/processed")
if not processed_dir.exists():
    processed_dir = Path("../../../data/processed")

snapshot_paths = sorted(
    p for p in processed_dir.glob("financials_*.parquet") if "timeseries" not in p.name
)
timeseries_paths = sorted(processed_dir.glob("financials_timeseries_*.xlsx"))
if not snapshot_paths:
    raise FileNotFoundError(
        f"No financials_*.parquet snapshot found in {processed_dir}"
    )
if not timeseries_paths:
    raise FileNotFoundError(
        f"No financials_timeseries_*.xlsx timeseries found in {processed_dir}"
    )

financials_df = pd.read_parquet(snapshot_paths[-1])
timeseries_df = pd.read_excel(timeseries_paths[-1], sheet_name="timeseries")

display(financials_df.head())
display(timeseries_df.head())

## Snapshot: `financials_<YYYY-MM-DD>.parquet`


In [ ]:
financials_df["as_of_date"] = pd.to_datetime(
    financials_df["as_of_date"], errors="coerce"
)

print("shape:", financials_df.shape)
print("index name:", financials_df.index.name)

snapshot_summary = pd.DataFrame(
    {
        "dtype": financials_df.dtypes.astype(str),
        "missing_pct": (financials_df.isna().mean() * 100).round(2),
    }
).sort_values("missing_pct", ascending=False)

snapshot_summary.head(20)

In [ ]:
top_missing = snapshot_summary.head(15).sort_values("missing_pct")

plt.figure(figsize=(10, 6))
sns.barplot(x=top_missing["missing_pct"], y=top_missing.index, color="#4C72B0")
plt.title("Snapshot: top missing columns")
plt.xlabel("Missing %")
plt.ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
top_industry = (
    financials_df["industry"]
    .fillna("MISSING")
    .astype(str)
    .value_counts()
    .head(15)
    .sort_values()
)
top_region = (
    financials_df["region"]
    .fillna("MISSING")
    .astype(str)
    .value_counts()
    .head(15)
    .sort_values()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.barplot(x=top_industry.values, y=top_industry.index, ax=axes[0], color="#55A868")
axes[0].set_title("Top industries")
axes[0].set_xlabel("count")
axes[0].set_ylabel("industry")

sns.barplot(x=top_region.values, y=top_region.index, ax=axes[1], color="#C44E52")
axes[1].set_title("Top regions")
axes[1].set_xlabel("count")
axes[1].set_ylabel("region")

plt.tight_layout()
plt.show()

In [ ]:
fields = [
    "Revenues",
    "NetIncomeLoss",
    "Assets",
    "Liabilities",
    "EBITDA",
    "LongTermDebt",
]

for col in fields:
    plt.figure(figsize=(10, 4))
    sns.histplot(financials_df[col].dropna(), bins=40, kde=True, color="#8172B3")
    plt.title(f"Snapshot distribution: {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()

## Timeseries: `financials_timeseries_<YYYY-MM-DD>.xlsx`


In [ ]:
timeseries_df["as_of_date"] = pd.to_datetime(
    timeseries_df["as_of_date"], errors="coerce"
)

print("shape:", timeseries_df.shape)

timeseries_summary = pd.DataFrame(
    {
        "dtype": timeseries_df.dtypes.astype(str),
        "missing_pct": (timeseries_df.isna().mean() * 100).round(2),
    }
).sort_values("missing_pct", ascending=False)

timeseries_summary.head(20)

In [ ]:
rows_by_year = timeseries_df["as_of_date"].dt.year.value_counts().sort_index()

plt.figure(figsize=(10, 4))
sns.lineplot(x=rows_by_year.index, y=rows_by_year.values, marker="o", color="#4C72B0")
plt.title("Timeseries: rows by as_of_date year")
plt.xlabel("year")
plt.ylabel("rows")
plt.tight_layout()
plt.show()

In [ ]:
example_cik = timeseries_df["cik"].iloc[0]
company_df = timeseries_df[timeseries_df["cik"] == example_cik].sort_values(
    "as_of_date"
)
example_ticker = company_df["ticker"].iloc[0]

fields = ["Revenues", "NetIncomeLoss", "Assets", "Liabilities"]
company_long = company_df.melt(
    id_vars=["as_of_date"],
    value_vars=fields,
    var_name="field",
    value_name="value",
)

plt.figure(figsize=(12, 5))
sns.lineplot(data=company_long, x="as_of_date", y="value", hue="field", marker="o")
plt.title(f"Example company: cik={example_cik} ticker={example_ticker}")
plt.xlabel("as_of_date")
plt.ylabel("value")
plt.tight_layout()
plt.show()